# Bloom Craft — เครื่องเจนรูปแยก (SDXL)

รันบน **Colab อีกบัญชี** (ไม่ใช่บัญชีที่รัน notebook วิเคราะห์) · Runtime → Change runtime type → **T4 GPU** → Run all

1. รอเซลล์สุดท้ายขึ้นลิงก์ `https://....trycloudflare.com`
2. คัดลอกลิงก์ไปใส่ `GEN_URL = "..."` ในข้อ 2) ตั้งค่า ของ notebook วิเคราะห์ (`flower_counter_demo_4ชนิด_v6_แบ่งโซน.ipynb`) แล้วค่อย Run all ฝั่งนั้น
3. เปิดเว็บจากลิงก์ของ notebook วิเคราะห์ตามปกติ — กด Generate แล้วงานเจนจะวิ่งมาที่นี่

ทำไมแยก: SDXL ใหญ่ ~7GB ถ้าอยู่เครื่องเดียวกับ SAM 3 แรม Colab เต็มจนแครช · เครื่องนี้ไม่ต้องใช้ HF token


In [ ]:
%pip install -q diffusers accelerate fastapi uvicorn

## 1) โหลด SDXL + IP-Adapter

In [ ]:
import base64, io, re, os, subprocess, threading, time
import torch
from PIL import Image, ImageDraw
from diffusers import AutoencoderKL, DPMSolverMultistepScheduler, StableDiffusionXLPipeline, StableDiffusionXLInpaintPipeline
from diffusers.image_processor import IPAdapterMaskProcessor
from transformers import CLIPVisionModelWithProjection

GEN_MODEL = "SG161222/RealVisXL_V5.0"   # SDXL สายภาพถ่ายจริง
GEN_STEPS = 20
GEN_W, GEN_H = 832, 1024   # แนวตั้ง (ขนาดที่ SDXL ถนัด)
REF_SCALE = 0.4            # ดอกชนิดเดียว
REF_SCALE_ZONE = 0.6       # หลายชนิด: รูปอ้างอิงคุมแค่โซนตัวเอง
REFINE_STRENGTH = 0.7      # รอบวาดซ้ำทีละโซน: สูง → เปลี่ยนทรงดอกได้มาก / ต่ำ → ขอบโซนกลืนกว่า
# shortcut: เว็บผ่าน Cloudflare ตัดที่ 100 วินาที → 4 ชนิด (4 โซน) บน T4 อาจเกิน; ถ้าเจอ error 524/502 ลด GEN_STEPS

assert torch.cuda.is_available(), "ต้องเปิด GPU: Runtime → Change runtime type → T4 GPU"
pipe = StableDiffusionXLPipeline.from_pretrained(
    GEN_MODEL, torch_dtype=torch.float16, variant="fp16", use_safetensors=True,
    vae=AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16),   # VAE เดิมใน fp16 ได้รูปดำ
    image_encoder=CLIPVisionModelWithProjection.from_pretrained(
        "h94/IP-Adapter", subfolder="models/image_encoder", torch_dtype=torch.float16))
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, algorithm_type="dpmsolver++", use_karras_sigmas=True)
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter-plus_sdxl_vit-h.safetensors")
pipe.set_progress_bar_config(disable=True)
pipe.to("cuda")   # ~9GB อยู่บน T4 ทั้งตัว (เครื่องนี้ไม่มี SAM 3 มาแย่ง)
inpaint = StableDiffusionXLInpaintPipeline(**pipe.components)   # ใช้โมเดลชุดเดียวกัน ไม่กิน GPU เพิ่ม
inpaint.set_progress_bar_config(disable=True)
print(f"พร้อม: {GEN_MODEL} · GPU ว่างเหลือ {torch.cuda.mem_get_info()[0] / 2**30:.1f}GB")


def zone_images(n, top=0.04, bottom=0.68):
    """n โซนแนวตั้งซ้าย→ขวา ครอบเฉพาะส่วนดอก (ภาชนะอยู่ล่าง) · ขอบซ้อนกันนิดให้กลมกลืน"""
    ims = []
    for i in range(n):
        m = Image.new("L", (GEN_W, GEN_H), 0)
        x0, x1 = max(0, GEN_W * i // n - 32), min(GEN_W, GEN_W * (i + 1) // n + 32)
        ImageDraw.Draw(m).rectangle((x0, int(GEN_H * top), x1, int(GEN_H * bottom)), fill=255)
        ims.append(m)
    return ims


def zone_masks(n):
    t = IPAdapterMaskProcessor().preprocess(zone_images(n), height=GEN_H, width=GEN_W)   # [n,1,H,W]
    return [t.reshape(1, n, t.shape[2], t.shape[3])]


_gpu = threading.Lock()


def generate(body):
    """body จาก notebook วิเคราะห์: prompt, negative, lead, refs=[{png(base64), plural, phrase}], seed"""
    refs = [(Image.open(io.BytesIO(base64.b64decode(r["png"]))).convert("RGB"), r) for r in body.get("refs") or []][:4]
    seed = int(body.get("seed") or time.time())
    neg = body.get("negative", "")
    if len(refs) >= 2:   # หลายชนิด → ชนิดละโซน กันดอกผสมกัน
        scale, ip, xkw = [[REF_SCALE_ZONE] * len(refs)], [[im for im, _ in refs]], {"ip_adapter_masks": zone_masks(len(refs))}
    else:
        scale, ip, xkw = (REF_SCALE if refs else 0.0), (refs[0][0] if refs else Image.new("RGB", (224, 224), "white")), None
    with _gpu, torch.inference_mode():
        pipe.set_ip_adapter_scale(scale)
        im = pipe(prompt=body["prompt"], negative_prompt=neg, ip_adapter_image=ip, cross_attention_kwargs=xkw,
                  num_inference_steps=GEN_STEPS, guidance_scale=6.0, width=GEN_W, height=GEN_H,
                  generator=torch.Generator("cpu").manual_seed(seed)).images[0]
        if len(refs) >= 2:   # วาดซ้ำทีละโซนด้วย prompt ของชนิดนั้นอย่างเดียว (คำใน prompt รวมลากทุกโซนเป็นทรงเดียวกัน)
            pipe.set_ip_adapter_scale(0.6)
            for i, ((r, meta), z) in enumerate(zip(refs, zone_images(len(refs)))):
                if not meta.get("phrase"):
                    continue
                others = ", ".join(m["plural"] for _, m in refs if m.get("plural") and m["plural"] != meta.get("plural"))
                im = inpaint(prompt=f"{body.get('lead', 'A flower arrangement')}, close-up of {meta['phrase']}. "
                                    "Florist photo, photorealistic",
                             negative_prompt=neg + ", " + others, image=im, mask_image=inpaint.mask_processor.blur(z, 32),
                             ip_adapter_image=r, strength=REFINE_STRENGTH, num_inference_steps=GEN_STEPS,
                             guidance_scale=6.0, width=GEN_W, height=GEN_H,
                             generator=torch.Generator("cpu").manual_seed(seed + 100 + i)).images[0]
    return im


## 2) ทดสอบ (ข้ามได้)

In [ ]:
# ทดสอบเจน 1 รูป (ไม่มีรูปอ้างอิง) — เช็กว่าโมเดลทำงาน
t0 = time.time()
test = generate({"prompt": "A hand-tied bouquet held in hands, stems wrapped in kraft paper with a ribbon, made of 7 pink tulips, "
                           "5 yellow sunflowers with big dark brown centers. Florist photo, plain light background, photorealistic",
                 "negative": "cartoon, drawing, blurry, deformed petals, text, watermark, hybrid flower", "seed": 0})
print(f"{time.time() - t0:.0f} วินาที")
test.resize((416, 512))

## 3) เปิดเซิร์ฟเวอร์ → เอาลิงก์ไปใส่ GEN_URL

In [ ]:
from fastapi import Body, FastAPI
from fastapi.responses import JSONResponse
import uvicorn

PORT = 8001
api = FastAPI()


@api.get("/")
def health():
    return {"ok": True, "model": GEN_MODEL}


@api.post("/generate")
def api_generate(body: dict = Body(...)):
    if not isinstance(body.get("prompt"), str) or not body["prompt"]:
        return JSONResponse({"error": "no_prompt"}, 400)
    t0 = time.time()
    im = generate(body)
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=90)
    print(f"[gen] {time.time() - t0:.0f} วินาที · {len(body.get('refs') or [])} รูปอ้างอิง · {body['prompt'][:90]}", flush=True)
    return {"image": "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()}


if globals().get("_server") is None:   # รันเซลล์ซ้ำ → ไม่เปิดพอร์ตซ้ำ
    _server = uvicorn.Server(uvicorn.Config(api, host="0.0.0.0", port=PORT, log_level="warning"))
    threading.Thread(target=_server.run, daemon=True).start()
    time.sleep(2)

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "-O", "cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("cloudflared", 0o755)
if globals().get("_tunnel_url") is None or _tun.poll() is not None:   # tunnel ตาย → เปิดใหม่ ได้ลิงก์ใหม่
    _tunnel_url = None
    _tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"], stderr=subprocess.PIPE, text=True)
    t0 = time.time()
    for line in _tun.stderr:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            _tunnel_url = m.group(0)
            break
        if time.time() - t0 > 60:
            break
    threading.Thread(target=lambda: [None for _ in _tun.stderr], daemon=True).start()   # อ่านทิ้ง กัน pipe เต็ม
print("🎨 GEN_URL =", repr(globals().get("_tunnel_url")), "← คัดลอกไปใส่ใน notebook วิเคราะห์ ข้อ 2) ตั้งค่า")
print("ปล่อยแท็บนี้เปิดไว้ระหว่างใช้งาน · log การเจนจะขึ้นด้านล่าง")